# LeetCode 324 — Wiggle Sort II bằng Python

Link đề: https://leetcode.com/problems/wiggle-sort-ii/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Quickselect, Divide and Conquer

Skeleton LeetCode:

```python
class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Bài có **nhiều đáp án đúng**, nên test **không so cứng** với một mảng mẫu mà kiểm tra **tính hợp lệ** (`nums[0] < nums[1] > nums[2] < ...`) và **cùng đa tập phần tử** với đầu vào; đối chiếu với một brute force độc lập (thử mọi hoán vị) về việc "có tồn tại đáp án".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sắp xếp lại mảng sao cho `nums[0] < nums[1] > nums[2] < nums[3] > ...` (so sánh **nghiêm ngặt** xen kẽ), sửa **tại chỗ**; đề đảm bảo luôn có đáp án.
2. Phân biệt với **Wiggle Sort I** (`<=`, `>=`): khi có **phần tử bằng nhau**, các mẹo hoán đổi cục bộ **không đủ**.
3. Đi từ **thử mọi hoán vị** → **sắp xếp rồi xen kẽ nửa nhỏ và nửa lớn, mỗi nửa đảo ngược** `O(n log n)` → **đếm (counting) vì `nums[i] ≤ 5000`** `O(n + V)` → **tìm trung vị bằng quickselect + phân hoạch 3 phần với chỉ số ảo** `O(n)` thời gian, `O(1)` bộ nhớ phụ (follow-up).
4. Hiểu **vì sao phải đảo ngược** từng nửa khi xen kẽ (để hai phần tử bằng nhau nằm cạnh nhau ở chỗ giao hai nửa **không** bị kề nhau) và **vì sao tách tại `⌈n/2⌉`**.
5. Tránh các lỗi kinh điển: xen kẽ **không đảo ngược**, tách nửa **sai chỗ** khi `n` lẻ, dùng hoán đổi cục bộ kiểu Wiggle Sort I, và kiểm tra test bằng **so sánh cứng** với một đáp án duy nhất.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, hãy sắp xếp lại sao cho `nums[0] < nums[1] > nums[2] < nums[3]...`

Bạn có thể giả định đầu vào luôn có đáp án hợp lệ.

**Ví dụ 1**

- Input: `nums = [1,5,1,1,6,4]`
- Output: `[1,6,1,5,1,4]`
- Giải thích: `[1,4,1,5,1,6]` cũng được chấp nhận.

**Ví dụ 2**

- Input: `nums = [1,3,2,2,3,1]`
- Output: `[2,3,1,3,1,2]`

**Ràng buộc:** `1 <= nums.length <= 5·10^4`; `0 <= nums[i] <= 5000`; đảm bảo có đáp án.

**Follow up:** làm được trong `O(n)` thời gian và/hoặc **tại chỗ** với `O(1)` bộ nhớ phụ không?

In [1]:
from typing import List
from collections import Counter
from itertools import permutations
import random
import time

examples = [
    [1, 5, 1, 1, 6, 4],
    [1, 3, 2, 2, 3, 1],
]


def is_wiggle(a):
    """nums[0] < nums[1] > nums[2] < nums[3] > ... (so sánh nghiêm ngặt)."""
    return all((a[i] < a[i + 1]) if i % 2 == 0 else (a[i] > a[i + 1]) for i in range(len(a) - 1))


def check_result(original, result):
    """Hợp lệ nếu là hoán vị của đầu vào và thỏa mẫu zigzag nghiêm ngặt."""
    return sorted(result) == sorted(original) and is_wiggle(result)


for nums in examples:
    print(nums, "| mẫu đáp án trong đề có hợp lệ?", is_wiggle([1, 6, 1, 5, 1, 4]) if nums[0] == 1 and nums[1] == 5 else is_wiggle([2, 3, 1, 3, 1, 2]))

[1, 5, 1, 1, 6, 4] | mẫu đáp án trong đề có hợp lệ? True
[1, 3, 2, 2, 3, 1] | mẫu đáp án trong đề có hợp lệ? True


## 3. Kiến thức nền

### 3.1. Mẫu zigzag nghiêm ngặt và hàm kiểm tra

Vị trí **chẵn** (`0, 2, 4, ...`) là **đáy** (nhỏ hơn cả hai hàng xóm), vị trí **lẻ** (`1, 3, ...`) là **đỉnh** (lớn hơn cả hai hàng xóm). Hàm `is_wiggle` kiểm tra từng cặp liền kề: ở chỉ số chẵn cần `a[i] < a[i+1]`, ở chỉ số lẻ cần `a[i] > a[i+1]`. Vì có **nhiều đáp án đúng**, ta kiểm tra bằng tính chất này thay vì so với một mảng cố định.
#### Code demo

In [2]:
print(is_wiggle([1, 6, 1, 5, 1, 4]), is_wiggle([1, 4, 1, 5, 1, 6]), is_wiggle([1, 1, 5, 6]), is_wiggle([2, 2]))

True True False False


### 3.2. Điều kiện tồn tại đáp án

Nếu một giá trị xuất hiện `f` lần, nó không thể có hai bản kề nhau (kề nhau thì bằng nhau, vi phạm `<`/`>` nghiêm ngặt), nên các bản của nó phải nằm ở các vị trí **không kề nhau**: tối đa `⌈n/2⌉` vị trí. Vậy **cần `f ≤ ⌈n/2⌉` với mọi giá trị** — nhưng điều kiện này **chỉ cần chứ chưa đủ** (ví dụ `[1,2,2,2,3]` có `f = 3 = ⌈5/2⌉` mà **không** có đáp án, vì ba số `2` phải ở các vị trí `0,2,4` mà hai hàng xóm còn lại là `1` và `3` không cùng lớn hơn `2`). Vì vậy đề phải **đảm bảo** đầu vào có đáp án; ta dùng brute force (mục 5) làm trọng tài "có đáp án không" trên mảng nhỏ.
#### Code demo

In [3]:
def necessary_condition(nums):                         # chỉ là điều kiện CẦN, không đủ
    return max(Counter(nums).values()) <= (len(nums) + 1) // 2


print(necessary_condition([1, 5, 1, 1, 6, 4]), necessary_condition([1, 1, 1, 2]), necessary_condition([2, 2]))   # True False False

True False False


### 3.3. Chia đôi mảng đã sắp xếp: nửa nhỏ và nửa lớn

Sau khi sắp xếp, chia mảng tại `m = (n + 1) // 2` (tức `⌈n/2⌉`): `small = sorted[:m]` (các giá trị nhỏ hơn hoặc bằng trung vị) và `large = sorted[m:]` (các giá trị lớn). Ta sẽ đặt `small` vào các vị trí **chẵn** (đáy) và `large` vào các vị trí **lẻ** (đỉnh). Số vị trí chẵn là `⌈n/2⌉` và số vị trí lẻ là `⌊n/2⌋` — khớp kích thước hai nửa.
#### Code demo

In [4]:
nums = [1, 5, 1, 1, 6, 4]
s = sorted(nums)
m = (len(s) + 1) // 2
print(s, "| small =", s[:m], "| large =", s[m:])
print("số vị trí chẵn:", len(range(0, len(s), 2)), "| số vị trí lẻ:", len(range(1, len(s), 2)))

[1, 1, 1, 4, 5, 6] | small = [1, 1, 1] | large = [4, 5, 6]
số vị trí chẵn: 3 | số vị trí lẻ: 3


### 3.4. Gán theo bước nhảy: `a[0::2]` và `a[1::2]`

`a[0::2]` là các vị trí chẵn; `a[1::2]` là các vị trí lẻ. Gán lát cắt có bước: `a[0::2] = small`, `a[1::2] = large` (độ dài phải khớp). Đảo ngược danh sách bằng `lst[::-1]`.
#### Code demo

In [5]:
a = [0] * 6
a[0::2] = [1, 2, 3]
a[1::2] = [9, 8, 7]
print(a)                    # [1, 9, 2, 8, 3, 7]
print([1, 2, 3][::-1])      # [3, 2, 1]

[1, 9, 2, 8, 3, 7]
[3, 2, 1]


### 3.5. Quickselect (tìm phần tử thứ `k`) và phân hoạch 3 phần

**Quickselect** tìm phần tử nhỏ thứ `k` trong `O(n)` trung bình bằng cách phân hoạch quanh một pivot ngẫu nhiên và chỉ đi vào một nửa. **Phân hoạch 3 phần (cờ Hà Lan)** chia mảng thành `< pivot`, `== pivot`, `> pivot` — xử lý tốt các giá trị trùng. Ta dùng chúng để tìm **trung vị** mà không sắp xếp toàn bộ.
#### Code demo

In [6]:
def quickselect(a, k, rng=random):
    """Trả phần tử nhỏ thứ k (k tính từ 0) của a, KHÔNG làm thay đổi a."""
    arr = list(a)
    while True:
        pivot = rng.choice(arr)
        less = [x for x in arr if x < pivot]
        equal = [x for x in arr if x == pivot]
        if k < len(less):
            arr = less
        elif k < len(less) + len(equal):
            return pivot
        else:
            k -= len(less) + len(equal)
            arr = [x for x in arr if x > pivot]


random.seed(324)
arr = [1, 5, 1, 1, 6, 4]
print(quickselect(arr, 2), sorted(arr))        # phần tử nhỏ thứ 2 (chỉ số 0): 1

1 [1, 1, 1, 4, 5, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`nums = [1, 5, 1, 1, 6, 4]` ⇒ sắp xếp `[1, 1, 1, 4, 5, 6]`; `small = [1, 1, 1]`, `large = [4, 5, 6]`.

- **Xen kẽ không đảo ngược:** `[1, 4, 1, 5, 1, 6]` — hợp lệ ở ví dụ này (đây chính là đáp án phụ của đề).
- Nhưng với `[4, 5, 5, 6]`: sắp xếp `[4, 5, 5, 6]`, `small = [4, 5]`, `large = [5, 6]`; xen kẽ `[4, 5, 5, 6]` — **không hợp lệ** (`5` bằng `5` ở hai vị trí kề nhau).
- **Đảo ngược từng nửa:** `small` ngược `[5, 4]`, `large` ngược `[6, 5]` ⇒ `[5, 6, 4, 5]` hợp lệ (`5 < 6 > 4 < 5`).

**Cách nghĩ:** hai phần tử bằng nhau có thể nằm ở **cuối `small`** và **đầu `large`** (quanh trung vị). Nếu đặt chúng tại hai vị trí **kề nhau** thì vi phạm. Đảo ngược mỗi nửa đẩy các bản trùng của trung vị **ra xa nhau** (một ở đầu mảng, một ở cuối).

## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng.** Với mỗi hoán vị của `nums` kiểm tra `is_wiggle`; trả hoán vị đầu tiên hợp lệ (chỉ dùng cho `n ≲ 8`).
#### Code demo

In [7]:
def wiggle_bruteforce(nums):
    for perm in permutations(nums):
        if is_wiggle(perm):
            nums[:] = perm
            return
    raise ValueError("không có đáp án")


for original in examples:
    a = list(original)
    wiggle_bruteforce(a)
    print(original, "->", a, "| hợp lệ:", check_result(original, a))

[1, 5, 1, 1, 6, 4] -> [1, 5, 1, 6, 1, 4] | hợp lệ: True
[1, 3, 2, 2, 3, 1] -> [1, 3, 2, 3, 1, 2] | hợp lệ: True


**Độ phức tạp (Cách 1).** **Time `O(n! · n)`**, **Space `O(n)`**; chỉ để làm oracle cho `n` rất nhỏ.

**Nút thắt.** Không khai thác cấu trúc: đáp án chỉ cần **nửa nhỏ ở vị trí chẵn, nửa lớn ở vị trí lẻ**.

## 6. Ý tưởng thuật toán / bước đột phá

**Đặt nửa nhỏ vào vị trí chẵn, nửa lớn vào vị trí lẻ, mỗi nửa theo thứ tự ngược:**

`nums[0::2] = small[::-1]`, `nums[1::2] = large[::-1]` (với `small`, `large` là hai nửa của mảng đã sắp, chia tại `⌈n/2⌉`).

Khi đó mỗi đỉnh (vị trí lẻ) thuộc `large` và mỗi đáy hàng xóm thuộc `small`; vì `small ≤ large` nên đỉnh `≥` đáy; còn **nghiêm ngặt** nhờ đảo ngược: chỉ khi giá trị bằng trung vị mới có thể bằng nhau, và đảo ngược đảm bảo hai bản đó không kề nhau (chứng minh mục 10).

Để **không cần sắp xếp** toàn bộ: chỉ cần tìm **trung vị** (quickselect `O(n)`) và phân hoạch ba phần với **chỉ số ảo** `(1 + 2·i) mod (n | 1)` đặt luôn các phần tử lớn vào vị trí lẻ, nhỏ vào vị trí chẵn, tại chỗ.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp, tách đôi, đảo ngược từng nửa, xen kẽ (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def wiggle_sort_halves(nums):
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    nums[0::2] = s[:m][::-1]                    # nửa nhỏ, đảo ngược, vào vị trí chẵn
    nums[1::2] = s[m:][::-1]                    # nửa lớn, đảo ngược, vào vị trí lẻ


for original in examples:
    a = list(original)
    wiggle_sort_halves(a)
    print(original, "->", a, "| hợp lệ:", check_result(original, a))

[1, 5, 1, 1, 6, 4] -> [1, 6, 1, 5, 1, 4] | hợp lệ: True
[1, 3, 2, 2, 3, 1] -> [2, 3, 1, 3, 1, 2] | hợp lệ: True


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        arr = sorted(nums)
        n = len(arr)
        i, j = (n - 1) >> 1, n - 1
        for k in range(n):
            if k % 2 == 0:
                nums[k] = arr[i]
                i -= 1
            else:
                nums[k] = arr[j]
                j -= 1


a = [1, 5, 1, 1, 6, 4]
Solution().wiggleSort(a)
print(a, is_wiggle(a))

[1, 6, 1, 5, 1, 4] True


(Bản nộp ở trên dùng hai con trỏ `i` (từ cuối nửa nhỏ) và `j` (từ cuối mảng) — chính là "đảo ngược từng nửa" viết gọn.)

**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** (bản sao đã sắp).

### Cách 3 — Sắp xếp bằng đếm (vì `nums[i] ≤ 5000`)

**Giải thích.** Thay `sorted` bằng **counting sort**: đếm số lần mỗi giá trị, dựng mảng đã sắp trong `O(n + V)` với `V = 5001`; phần còn lại giống Cách 2.
#### Code demo

In [10]:
def wiggle_sort_counting(nums, max_value=5000):
    counts = [0] * (max_value + 1)
    for x in nums:
        counts[x] += 1
    s = []
    for v, c in enumerate(counts):
        s.extend([v] * c)
    m = (len(nums) + 1) // 2
    nums[0::2] = s[:m][::-1]
    nums[1::2] = s[m:][::-1]


for original in examples:
    a = list(original)
    wiggle_sort_counting(a)
    print(original, "->", a, "| hợp lệ:", check_result(original, a))

[1, 5, 1, 1, 6, 4] -> [1, 6, 1, 5, 1, 4] | hợp lệ: True
[1, 3, 2, 2, 3, 1] -> [2, 3, 1, 3, 1, 2] | hợp lệ: True


**Độ phức tạp Cách 3:** **Time `O(n + V)`** với `V = 5001`, **Space `O(n + V)`**. Nhanh hơn `sorted` khi `n` rất lớn và `V` nhỏ.

### Cách 4 — Trung vị bằng quickselect + phân hoạch 3 phần với chỉ số ảo (`O(n)`, `O(1)` phụ)

**Giải thích.**

1. Tìm trung vị `mid` (phần tử nhỏ thứ `⌊n/2⌋`) bằng quickselect.
2. Định nghĩa **chỉ số ảo** `A(i) = nums[(1 + 2·i) mod (n | 1)]`: duyệt `i = 0, 1, 2, ...` thì chạm các vị trí `1, 3, 5, ..., ` rồi `0, 2, 4, ...` (các vị trí lẻ trước, rồi các vị trí chẵn).
3. Chạy phân hoạch **ba phần** trên dãy ảo: các phần tử **`> mid` dồn về đầu** (tức các vị trí lẻ), **`< mid` dồn về cuối** (các vị trí chẵn cuối), bằng `mid` ở giữa.
#### Code demo

In [11]:
def wiggle_sort_virtual(nums, rng=random):
    n = len(nums)
    if n < 2:
        return
    mid = quickselect(nums, n // 2, rng)                # trung vị (không đổi nums)
    size = n | 1                                         # số lẻ >= n: làm modulo để ánh xạ vị trí ảo

    def idx(i):
        return (1 + 2 * i) % size

    lo, i, hi = 0, 0, n - 1
    while i <= hi:
        v = nums[idx(i)]
        if v > mid:                                       # lớn hơn trung vị: đưa về phía đầu (vị trí lẻ)
            nums[idx(lo)], nums[idx(i)] = nums[idx(i)], nums[idx(lo)]
            lo += 1
            i += 1
        elif v < mid:                                     # nhỏ hơn trung vị: đưa về phía cuối (vị trí chẵn)
            nums[idx(i)], nums[idx(hi)] = nums[idx(hi)], nums[idx(i)]
            hi -= 1
        else:
            i += 1


random.seed(3240)
for original in examples:
    a = list(original)
    wiggle_sort_virtual(a)
    print(original, "->", a, "| hợp lệ:", check_result(original, a))

[1, 5, 1, 1, 6, 4] -> [1, 5, 1, 6, 1, 4] | hợp lệ: True
[1, 3, 2, 2, 3, 1] -> [2, 3, 1, 3, 1, 2] | hợp lệ: True


**Độ phức tạp Cách 4:** quickselect **`O(n)` trung bình** (bản trong notebook dùng danh sách phụ nên `O(n)` bộ nhớ; bản tại chỗ thật dùng phân hoạch 3 phần tại chỗ để `O(1)`), phân hoạch ảo **`O(n)`**; tổng **Time `O(n)` trung bình**, **Space `O(1)` phụ** (với quickselect tại chỗ). Là lời giải cho follow-up.

> Cách 2 là bản nộp thông dụng. Cách 3 tận dụng miền giá trị nhỏ. Cách 4 trả lời follow-up (khó và tinh vi).

## 8. TRACE từng bước

Ta trace Cách 2 trên `[1, 5, 1, 1, 6, 4]` (và ca "trùng trung vị" `[4, 5, 5, 6]`) và Cách 4 trên `[1, 3, 2, 2, 3, 1]` với chỉ số ảo.
#### Code demo

In [12]:
def trace_halves(nums):
    print("nums =", nums)
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    small, large = s[:m], s[m:]
    print(f"  sắp xếp = {s}; small = {small}, large = {large}")
    print(f"  đảo ngược: small[::-1] = {small[::-1]}, large[::-1] = {large[::-1]}")
    out = [0] * len(nums)
    out[0::2] = small[::-1]
    out[1::2] = large[::-1]
    print(f"  vị trí chẵn <- small đảo, vị trí lẻ <- large đảo => {out}; hợp lệ: {is_wiggle(out)}")
    naive = [0] * len(nums)
    naive[0::2] = small
    naive[1::2] = large
    print(f"  (so sánh: xen kẽ KHÔNG đảo ngược => {naive}; hợp lệ: {is_wiggle(naive)})")
    return out


trace_halves([1, 5, 1, 1, 6, 4])
print()
trace_halves([4, 5, 5, 6])

nums = [1, 5, 1, 1, 6, 4]
  sắp xếp = [1, 1, 1, 4, 5, 6]; small = [1, 1, 1], large = [4, 5, 6]
  đảo ngược: small[::-1] = [1, 1, 1], large[::-1] = [6, 5, 4]
  vị trí chẵn <- small đảo, vị trí lẻ <- large đảo => [1, 6, 1, 5, 1, 4]; hợp lệ: True
  (so sánh: xen kẽ KHÔNG đảo ngược => [1, 4, 1, 5, 1, 6]; hợp lệ: True)

nums = [4, 5, 5, 6]
  sắp xếp = [4, 5, 5, 6]; small = [4, 5], large = [5, 6]
  đảo ngược: small[::-1] = [5, 4], large[::-1] = [6, 5]
  vị trí chẵn <- small đảo, vị trí lẻ <- large đảo => [5, 6, 4, 5]; hợp lệ: True
  (so sánh: xen kẽ KHÔNG đảo ngược => [4, 5, 5, 6]; hợp lệ: False)


[5, 6, 4, 5]

In [13]:
def trace_virtual(nums):
    n = len(nums)
    arr = list(nums)
    mid = sorted(arr)[n // 2]
    size = n | 1
    idx = lambda i: (1 + 2 * i) % size
    print("nums =", arr, "| trung vị =", mid, "| ánh xạ i -> vị trí thật:", [idx(i) for i in range(n)])
    lo, i, hi = 0, 0, n - 1
    while i <= hi:
        v = arr[idx(i)]
        action = "lớn hơn trung vị: đổi với lo, lo++, i++" if v > mid else ("nhỏ hơn trung vị: đổi với hi, hi--" if v < mid else "bằng trung vị: i++")
        if v > mid:
            arr[idx(lo)], arr[idx(i)] = arr[idx(i)], arr[idx(lo)]
            lo += 1
            i += 1
        elif v < mid:
            arr[idx(i)], arr[idx(hi)] = arr[idx(hi)], arr[idx(i)]
            hi -= 1
        else:
            i += 1
        print(f"  v={v}: {action:<44} -> arr={arr}")
    print("  kết quả:", arr, "| hợp lệ:", is_wiggle(arr))
    return arr


trace_virtual([1, 3, 2, 2, 3, 1])

nums = [1, 3, 2, 2, 3, 1] | trung vị = 2 | ánh xạ i -> vị trí thật: [1, 3, 5, 0, 2, 4]
  v=3: lớn hơn trung vị: đổi với lo, lo++, i++      -> arr=[1, 3, 2, 2, 3, 1]
  v=2: bằng trung vị: i++                           -> arr=[1, 3, 2, 2, 3, 1]
  v=1: nhỏ hơn trung vị: đổi với hi, hi--           -> arr=[1, 3, 2, 2, 1, 3]
  v=3: lớn hơn trung vị: đổi với lo, lo++, i++      -> arr=[1, 3, 2, 3, 1, 2]
  v=1: nhỏ hơn trung vị: đổi với hi, hi--           -> arr=[2, 3, 1, 3, 1, 2]
  v=2: bằng trung vị: i++                           -> arr=[2, 3, 1, 3, 1, 2]
  kết quả: [2, 3, 1, 3, 1, 2] | hợp lệ: True


[2, 3, 1, 3, 1, 2]

Bảng tóm tắt (`[1, 5, 1, 1, 6, 4]`):

| bước | giá trị |
|---|---|
| sắp xếp | `[1, 1, 1, 4, 5, 6]` |
| `small` / `large` | `[1, 1, 1]` / `[4, 5, 6]` |
| đảo ngược | `[1, 1, 1]` / `[6, 5, 4]` |
| xen kẽ (chẵn ← small, lẻ ← large) | `[1, 6, 1, 5, 1, 4]` — đúng đáp án mẫu của đề |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (hai con trỏ). Đúng `class Solution`, đúng `wiggleSort(self, nums: List[int]) -> None`, sửa `nums` **tại chỗ**.
#### Code demo

In [14]:
class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        arr = sorted(nums)
        n = len(arr)
        i, j = (n - 1) >> 1, n - 1                 # i: cuối nửa nhỏ (đọc ngược), j: cuối mảng (đọc ngược nửa lớn)
        for k in range(n):
            if k % 2 == 0:                         # vị trí chẵn: đáy, lấy từ nửa nhỏ theo thứ tự ngược
                nums[k] = arr[i]
                i -= 1
            else:                                  # vị trí lẻ: đỉnh, lấy từ nửa lớn theo thứ tự ngược
                nums[k] = arr[j]
                j -= 1


sol = Solution()
for original in examples:
    a = list(original)
    sol.wiggleSort(a)
    assert check_result(original, a), (original, a)
print("OK: 2 ví dụ của đề (kiểm tra hợp lệ + cùng đa tập)")

OK: 2 ví dụ của đề (kiểm tra hợp lệ + cùng đa tập)


## 10. Vì sao thuật toán đúng

Gọi `s` là mảng đã sắp, `m = ⌈n/2⌉`, `small = s[:m]`, `large = s[m:]`.

- **Phát biểu:** nếu mảng **có** ít nhất một đáp án (đề đảm bảo) thì cách xen kẽ đảo ngược cũng cho một đáp án. Xét một cặp kề `(đáy, đỉnh)`: đáy thuộc `small`, đỉnh thuộc `large` ⇒ `đáy ≤ đỉnh` (vì `small ≤ large` trong mảng sắp). Chỉ có thể **bằng nhau** nếu cả hai bằng giá trị ranh giới `v = s[m-1] = s[m]`; việc còn lại là chứng minh sự đảo ngược khiến hai bản `v` như vậy không bao giờ kề nhau.
- **Vị trí của các bản `v`:** trong `small[::-1]` (đặt tại `0, 2, 4, ...`) các bản `v` nằm ở **đầu** (vì `v` là phần tử lớn nhất của `small`), tức các vị trí chẵn nhỏ nhất `0, 2, ...`; trong `large[::-1]` (đặt tại `1, 3, ...`) các bản `v` nằm ở **cuối** (vì `v` là phần tử nhỏ nhất của `large`), tức các vị trí lẻ lớn nhất. Hai nhóm vị trí này nằm ở **hai đầu đối diện** của mảng; chúng chỉ có thể gặp nhau khi có quá nhiều bản `v` (khi đó mọi cách sắp xếp đều bị `v` kề `v`, tức **không có đáp án** — trái với đảm bảo của đề). Phần chứng minh hình thức dài, nên ta **kiểm chứng bằng thực nghiệm** (mục 12.2): với mọi mảng nhỏ mà brute force tìm được đáp án thì cách xen kẽ đảo ngược cũng cho đáp án hợp lệ.
- **Không đảo ngược sẽ sai:** các bản `v` ở **cuối** `small` (chẵn lớn) và **đầu** `large` (lẻ nhỏ) sẽ ở các vị trí kề nhau ở giữa mảng.
- **Cách 4:** phân hoạch ba phần với chỉ số ảo đặt các phần tử `> mid` vào các vị trí lẻ trước (từ `1`), các phần tử `< mid` vào các vị trí chẵn cuối (từ cuối dãy ảo), còn bằng `mid` ở giữa dãy ảo: cùng cấu trúc "lớn ở vị trí lẻ gần đầu, nhỏ ở vị trí chẵn gần cuối" như Cách 2 và cũng có tính chất không-kề cho các bản của trung vị.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (≤ `5·10^4`), `V = 5001`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi hoán vị | `O(n!·n)` | `O(n)` |
| 2. Sắp xếp + xen kẽ đảo ngược | `O(n log n)` | `O(n)` |
| 3. Counting sort + xen kẽ | `O(n + V)` | `O(n + V)` |
| 4. Quickselect + phân hoạch ảo | `O(n)` trung bình | `O(1)` phụ (quickselect tại chỗ) |

Với `n = 5·10^4`: Cách 2 cỡ `8·10^5`, Cách 3 cỡ `5.5·10^4`, Cách 4 cỡ `10^5` phép. Cách 4 trung bình tuyến tính (quickselect xấu nhất `O(n²)` rất hiếm với pivot ngẫu nhiên).

## 12. Bộ test

Vì có **nhiều đáp án đúng**, ta kiểm tra bằng `check_result`: (1) kết quả là **hoán vị** của đầu vào (cùng đa tập) và (2) thỏa **zigzag nghiêm ngặt**. `oracle_exists` là brute force độc lập: **thử mọi hoán vị** để biết đầu vào có đáp án không. Đầu vào ngẫu nhiên lớn được **sinh bằng cách dựng sẵn một dãy zigzag rồi xáo trộn**, nên chắc chắn có đáp án.
#### Code demo

In [15]:
def oracle_exists(nums):
    return any(is_wiggle(p) for p in set(permutations(nums)))


impls = {
    "bruteforce": wiggle_bruteforce,
    "halves": wiggle_sort_halves,
    "counting": wiggle_sort_counting,
    "virtual": wiggle_sort_virtual,
    "submit": sol.wiggleSort,
}

for original in examples:
    assert oracle_exists(original) and necessary_condition(original)
    for key, fn in impls.items():
        a = list(original)
        fn(a)
        assert check_result(original, a), (key, original, a)
print("PASS ví dụ của đề (mọi cách cho kết quả hợp lệ, cùng đa tập)")

PASS ví dụ của đề (mọi cách cho kết quả hợp lệ, cùng đa tập)


### 12.1. Các trường hợp biên
#### Code demo

In [16]:
edge_cases = [
    ([7], "một phần tử"),
    ([1, 2], "hai phần tử khác nhau"),
    ([2, 1], "hai phần tử đảo"),
    ([1, 2, 3], "ba phần tử tăng dần"),
    ([3, 2, 1], "ba phần tử giảm dần"),
    ([1, 1, 2], "n lẻ: hai bản bằng nhau (1 < 2 > 1)"),
    ([4, 5, 5, 6], "ca cần đảo ngược"),
    ([1, 1, 2, 2, 3, 3], "mỗi giá trị hai bản"),
    ([1, 2, 2, 3, 3], "giá trị 2 và 3 mỗi giá trị hai bản, n lẻ"),
    ([0, 5000, 0, 5000, 0], "giá trị biên 0 và 5000"),
    ([5, 5, 5, 1, 1, 1], "hai giá trị mỗi giá trị n/2 lần"),
    ([1, 3, 2, 2, 3, 1], "ví dụ 2"),
]
for original, label in edge_cases:
    assert necessary_condition(original)
    if len(original) <= 7:
        assert oracle_exists(original), label
    for key, fn in impls.items():
        if key == "bruteforce" and len(original) > 7:
            continue
        a = list(original)
        fn(a)
        assert check_result(original, a), (key, label, original, a)
    print("PASS", label)

PASS một phần tử
PASS hai phần tử khác nhau
PASS hai phần tử đảo
PASS ba phần tử tăng dần
PASS ba phần tử giảm dần
PASS n lẻ: hai bản bằng nhau (1 < 2 > 1)
PASS ca cần đảo ngược
PASS mỗi giá trị hai bản
PASS giá trị 2 và 3 mỗi giá trị hai bản, n lẻ
PASS giá trị biên 0 và 5000
PASS hai giá trị mỗi giá trị n/2 lần
PASS ví dụ 2


### 12.2. Tính chất và phản ví dụ

- Với mọi mảng nhỏ **có** đáp án (theo brute force `oracle_exists`), cách xen kẽ đảo ngược cũng cho đáp án hợp lệ; và mảng **không** có đáp án thì đương nhiên không cách nào làm được.
- Mọi cách cho kết quả là **hoán vị** của đầu vào và **thỏa zigzag**.
- **Phản ví dụ** cho xen kẽ không đảo ngược: `[4, 5, 5, 6]` cho `[4, 5, 5, 6]` (không hợp lệ).
- **Phản ví dụ** cho mẹo Wiggle Sort I (đổi chỗ cục bộ với `<=`): với `[4,5,5,6]` bản đó giữ nguyên cặp `5, 5` bằng nhau.
#### Code demo

In [17]:
rng = random.Random(324)
checked = 0
for _ in range(300):
    n = rng.randint(1, 7)
    nums = [rng.randint(0, 3) for _ in range(n)]
    if oracle_exists(nums):
        a = nums[:]
        wiggle_sort_halves(a)
        assert check_result(nums, a), nums
        checked += 1
    else:
        pass                                                      # không có đáp án: đề không bao giờ đưa vào
print("PASS: trên 300 mảng nhỏ, hễ brute force tìm được đáp án thì xen kẽ đảo ngược cũng hợp lệ (", checked, "mảng có đáp án )")


def sai_xen_ke_khong_dao(nums):
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    nums[0::2] = s[:m]                                           # SAI: không đảo ngược
    nums[1::2] = s[m:]


bad = [4, 5, 5, 6]
a = bad[:]
sai_xen_ke_khong_dao(a)
print("phản ví dụ [4,5,5,6]: không đảo ngược ->", a, "| hợp lệ:", is_wiggle(a))
assert not is_wiggle(a)


def sai_wiggle_I(nums):
    for i in range(len(nums) - 1):                              # SAI: mẹo Wiggle Sort I (<=, >=) với so sánh cục bộ
        if (i % 2 == 0 and nums[i] > nums[i + 1]) or (i % 2 == 1 and nums[i] < nums[i + 1]):
            nums[i], nums[i + 1] = nums[i + 1], nums[i]


b = [4, 5, 5, 6]
sai_wiggle_I(b)
print("Wiggle Sort I trên [4,5,5,6]:", b, "| hợp lệ (nghiêm ngặt):", is_wiggle(b))
assert not is_wiggle(b)

PASS: trên 300 mảng nhỏ, hễ brute force tìm được đáp án thì xen kẽ đảo ngược cũng hợp lệ ( 240 mảng có đáp án )
phản ví dụ [4,5,5,6]: không đảo ngược -> [4, 5, 5, 6] | hợp lệ: False
Wiggle Sort I trên [4,5,5,6]: [4, 5, 5, 6] | hợp lệ (nghiêm ngặt): False


### 12.3. Kiểm tra hợp lệ ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [18]:
def random_valid_input(n, hi, rng):
    """Sinh mảng CHẮC CHẮN có đáp án: dựng sẵn một dãy zigzag nghiêm ngặt trong [0, hi] rồi xáo trộn."""
    assert hi >= 1
    a = [rng.randint(0, hi - 1)]
    for i in range(1, n):
        if i % 2 == 1:                                           # đỉnh: lớn hơn đáy trước đó
            a.append(rng.randint(a[-1] + 1, hi))
        else:                                                    # đáy: nhỏ hơn đỉnh trước đó
            a.append(rng.randint(0, a[-1] - 1))
    assert is_wiggle(a)
    rng.shuffle(a)
    return a


rng = random.Random(4324)
for trial in range(3000):
    n = rng.randint(1, 40)
    hi = rng.choice([1, 2, 5, 50, 5000])
    nums = random_valid_input(n, hi, rng)
    for key, fn in impls.items():
        if key == "bruteforce":
            continue
        a = list(nums)
        if key == "virtual":
            random.seed(trial)
        fn(a)
        assert check_result(nums, a), (key, nums, a)
print("PASS 3000 mảng ngẫu nhiên: mọi cách (trừ vét cạn) cho kết quả hợp lệ")

# n = 5 * 10^4 và các dạng đặc biệt
big_cases = [
    ("ngẫu nhiên, giá trị tới 5000", random_valid_input(50_000, 5000, rng)),
    ("hai giá trị (0 và 1)", random_valid_input(50_000, 1, rng)),
    ("ít giá trị (0..2)", random_valid_input(50_000, 2, rng)),
    ("nhiều bản trùng (0..10)", random_valid_input(50_000, 10, rng)),
    ("n lẻ, giá trị 0..3", random_valid_input(49_999, 3, rng)),
    ("hai giá trị mỗi giá trị n/2 lần", [1, 5000] * 25_000),
]
for label, nums in big_cases:
    row = []
    for key in ("halves", "counting", "virtual", "submit"):
        a = list(nums)
        t0 = time.perf_counter()
        impls[key](a)
        dt = (time.perf_counter() - t0) * 1000
        assert check_result(nums, a), (label, key)
        row.append(f"{key}={dt:7.1f}ms")
    print(f"{label:<26} | " + " ".join(row))
print("PASS n = 5*10^4")

PASS 3000 mảng ngẫu nhiên: mọi cách (trừ vét cạn) cho kết quả hợp lệ


ngẫu nhiên, giá trị tới 5000 | halves=    9.2ms counting=    2.6ms virtual=   52.1ms submit=   17.4ms
hai giá trị (0 và 1)       | halves=    2.1ms counting=    2.3ms virtual=   14.4ms submit=    4.1ms
ít giá trị (0..2)          | halves=    2.8ms counting=    2.2ms virtual=   18.7ms submit=    5.3ms
nhiều bản trùng (0..10)    | halves=    4.0ms counting=    5.5ms virtual=   21.4ms submit=    6.4ms


n lẻ, giá trị 0..3         | halves=    3.0ms counting=    2.2ms virtual=   21.8ms submit=    5.4ms
hai giá trị mỗi giá trị n/2 lần | halves=    1.3ms counting=    2.4ms virtual=   13.6ms submit=    3.7ms
PASS n = 5*10^4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Xen kẽ nửa nhỏ và nửa lớn mà không đảo ngược
Xem `sai_xen_ke_khong_dao`: `[4, 5, 5, 6]` cho `[4, 5, 5, 6]` (hai số `5` kề nhau). **Vì sao sai:** các bản của trung vị ở cuối `small` và đầu `large` bị đặt kề nhau. **Code đúng:** đảo ngược từng nửa (hoặc lấy từ cuối về đầu).

### Lỗi 2: Tách nửa sai chỗ khi `n` lẻ
#### Code sai

In [19]:
def sai_tach_n_le(nums):
    s = sorted(nums)
    m = len(nums) // 2                                           # SAI: phải là (n + 1) // 2 (số vị trí chẵn)
    try:
        nums[0::2] = s[:m][::-1]
        nums[1::2] = s[m:][::-1]
    except ValueError as e:
        return f"ValueError: {e}"
    return nums


print("n = 5:", sai_tach_n_le([1, 5, 1, 1, 6]))
a = [1, 5, 1, 1, 6]
sol.wiggleSort(a)
print("đúng :", a, is_wiggle(a))

n = 5: ValueError: attempt to assign sequence of size 2 to extended slice of size 3
đúng : [1, 6, 1, 5, 1] True


**Vì sao sai:** có `⌈n/2⌉` vị trí chẵn và `⌊n/2⌋` vị trí lẻ; nửa nhỏ phải có `⌈n/2⌉` phần tử. Chia ở `n // 2` làm lệch kích thước (lỗi gán lát cắt có bước).

### Lỗi 3: Dùng mẹo hoán đổi cục bộ (Wiggle Sort I)
Xem `sai_wiggle_I`: với phần tử bằng nhau, đổi chỗ kề nhau không đủ để có `<`/`>` **nghiêm ngặt**. **Code đúng:** cách chia đôi + xen kẽ đảo ngược.

### Lỗi 4: Test bằng so sánh cứng với đáp án mẫu
#### Code sai

In [20]:
def test_cung(original, expected_sample):
    a = list(original)
    sol.wiggleSort(a)
    return a == expected_sample                                 # SAI: đề chấp nhận nhiều đáp án


print("so cứng với [1,4,1,5,1,6] (cũng là đáp án đúng):", test_cung([1, 5, 1, 1, 6, 4], [1, 4, 1, 5, 1, 6]))
a = [1, 5, 1, 1, 6, 4]
sol.wiggleSort(a)
print("kết quả của ta:", a, "| nhưng vẫn hợp lệ:", check_result([1, 5, 1, 1, 6, 4], a))

so cứng với [1,4,1,5,1,6] (cũng là đáp án đúng): False
kết quả của ta: [1, 6, 1, 5, 1, 4] | nhưng vẫn hợp lệ: True


**Vì sao sai:** bài có **nhiều đáp án**; phải kiểm tra **tính chất** (hoán vị + zigzag) chứ không so sánh một mảng cố định.

### Lỗi 5: Không giữ nguyên đa tập (làm mất/trùng phần tử)
#### Code sai

In [21]:
def sai_mat_phan_tu(nums):
    s = sorted(set(nums))                                        # SAI: bỏ trùng làm mất phần tử
    m = (len(s) + 1) // 2
    out = [0] * len(s)
    out[0::2] = s[:m][::-1]
    out[1::2] = s[m:][::-1]
    return out


r = sai_mat_phan_tu([1, 5, 1, 1, 6, 4])
print(r, "| cùng đa tập với đầu vào?", sorted(r) == sorted([1, 5, 1, 1, 6, 4]))

[4, 6, 1, 5] | cùng đa tập với đầu vào? False


**Vì sao sai:** kết quả phải là **hoán vị** của đầu vào (cùng số lần xuất hiện mỗi giá trị); bỏ trùng làm mảng ngắn đi.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**sắp xếp lại thành zigzag nghiêm ngặt**" → **sắp xếp, chia đôi tại `⌈n/2⌉`, xen kẽ với mỗi nửa đảo ngược**.
- Điều kiện CẦN để có đáp án: **không giá trị nào xuất hiện quá `⌈n/2⌉` lần** (chưa đủ — đề đảm bảo có đáp án).
- Giá trị nhỏ (`≤ 5000`) → **counting sort** thay `sorted`.
- Follow-up `O(n)` / `O(1)`: **quickselect tìm trung vị + phân hoạch ba phần với chỉ số ảo** `(1 + 2i) % (n | 1)`.
- Test bài nhiều đáp án: **hàm kiểm tra tính hợp lệ**, không so cứng.
- Họ hàng: **Wiggle Sort** (280, `<=`/`>=`), **Wiggle Subsequence** (376), **Sort Colors** (75, phân hoạch ba phần), **Kth Largest Element** (215, quickselect).
- Thử: `[4,5,5,6]`, `n` lẻ, hai giá trị mỗi giá trị `n/2` lần.

## 15. Biến thể của bài

1. **Wiggle Sort** (LeetCode 280): `nums[0] <= nums[1] >= nums[2] <= ...` — một lượt hoán đổi cục bộ `O(n)`.
2. **Wiggle Subsequence** (376): dãy con zigzag dài nhất.
3. **Kiểm tra có tồn tại đáp án** — chỉ brute force hoặc suy luận phức tạp hơn; điều kiện tần suất mới là điều kiện cần (`necessary_condition`).
4. **Zigzag ngược** (`>` rồi `<`): đảo vai nửa nhỏ/lớn.
5. **Sắp xếp để hiệu hai phần tử kề lớn nhất / nhỏ nhất**: bài tối ưu khác.
#### Code demo

In [22]:
def wiggle_sort_I(nums):
    """LeetCode 280: nums[0] <= nums[1] >= nums[2] <= ... (cho phép bằng nhau)."""
    for i in range(len(nums) - 1):
        if (i % 2 == 0) == (nums[i] > nums[i + 1]):
            nums[i], nums[i + 1] = nums[i + 1], nums[i]


def wiggle_length(nums):
    """LeetCode 376: độ dài dãy con zigzag dài nhất (O(n))."""
    if len(nums) < 2:
        return len(nums)
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            up = down + 1
        elif nums[i] < nums[i - 1]:
            down = up + 1
    return max(up, down)


a = [3, 5, 2, 1, 6, 4]
wiggle_sort_I(a)
print(a, all((a[i] <= a[i + 1]) if i % 2 == 0 else (a[i] >= a[i + 1]) for i in range(len(a) - 1)))
print(wiggle_length([1, 7, 4, 9, 2, 5]), wiggle_length([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]))   # 6 7

[3, 5, 1, 6, 2, 4] True
6 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử hoán vị | duyệt `n!` hoán vị | `O(n!·n)` | `O(n)` | oracle `n ≤ 8` |
| 2. Sắp xếp + xen kẽ đảo ngược | chia tại `⌈n/2⌉` | `O(n log n)` | `O(n)` | **bản nộp chuẩn** |
| 3. Counting sort | miền giá trị `≤ 5000` | `O(n + V)` | `O(n + V)` | `n` rất lớn |
| 4. Quickselect + chỉ số ảo | trung vị + phân hoạch 3 phần | `O(n)` TB | `O(1)` phụ | follow-up |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `is_wiggle([1, 3, 2])`; (b) `is_wiggle([1, 2, 2, 3])`; (c) `necessary_condition([1, 1, 1])`; (d) `sorted([5, 1, 4])[:2][::-1]`.

**Bài 2 (tính bằng tay).** Dùng Cách 2 cho `nums = [1, 3, 2, 2, 3, 1]`: sắp xếp, `small`, `large`, rồi xen kẽ đảo ngược; kiểm tra mẫu zigzag.

**Bài 3 (điền TODO).** Hoàn thiện `wiggle_todo(nums)` theo Cách 2.
#### Code demo

In [23]:
def wiggle_todo(nums):
    # TODO 1: s = sorted(nums); m = (len(nums) + 1) // 2
    # TODO 2: nums[0::2] = nửa nhỏ s[:m] đảo ngược
    # TODO 3: nums[1::2] = nửa lớn s[m:] đảo ngược
    pass


# Tự làm xong rồi mới mở test:
# for original in examples:
#     a = list(original); wiggle_todo(a); assert check_result(original, a)

**Bài 4 (tự viết lại).** Viết `wiggle_sort_reverse(nums)` tạo zigzag **đảo** (`nums[0] > nums[1] < nums[2] > ...`) bằng cách đổi vai hai nửa; kiểm tra bằng một hàm kiểm tra tương ứng.

**Bài 5 (biến thể / mini challenge).** Viết `random_zigzag(n, hi, seed)` sinh mảng độ dài `n`, giá trị trong `[0, hi]`, **chắc chắn có** đáp án zigzag nghiêm ngặt (dựng sẵn dãy zigzag rồi xáo trộn). Kiểm tra bằng `oracle_exists` cho `n ≤ 7`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: đáy `<` đỉnh `>` đáy; điều kiện tần suất chỉ là điều kiện cần. Bài 2: chia tại `m = 3`. Bài 5: dựng đáy/đỉnh xen kẽ rồi `shuffle`.
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `False` (`3 > ⌈3/2⌉ = 2`), `[4, 1]`. Bài 2: sắp `[1,1,2,2,3,3]`, `small = [1,1,2]`, `large = [2,3,3]`; kết quả `[2, 3, 1, 3, 1, 2]` (đúng đáp án mẫu). Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 4: sắp xếp **giảm dần** `t`, rồi `nums[0::2] = t[:m][::-1]`, `nums[1::2] = t[m:][::-1]` với `m = (n+1)//2` (soi gương Cách 2). Bài 5: phần tử đầu `a[0] ∈ [0, hi-1]`; ở vị trí lẻ chọn `a[i] ∈ [a[i-1]+1, hi]`, ở vị trí chẵn chọn `a[i] ∈ [0, a[i-1]-1]`; cuối cùng `rng.shuffle`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert is_wiggle([1, 3, 2]) is True and is_wiggle([1, 2, 2, 3]) is False
assert necessary_condition([1, 1, 1]) is False and sorted([5, 1, 4])[:2][::-1] == [4, 1]

# Bài 2
b2 = [1, 3, 2, 2, 3, 1]
s2 = sorted(b2)
m2 = (len(b2) + 1) // 2
out2 = [0] * 6
out2[0::2] = s2[:m2][::-1]
out2[1::2] = s2[m2:][::-1]
assert out2 == [2, 3, 1, 3, 1, 2] and is_wiggle(out2)


# Bài 3
def wiggle_todo_ans(nums):
    s = sorted(nums)
    m = (len(nums) + 1) // 2
    nums[0::2] = s[:m][::-1]
    nums[1::2] = s[m:][::-1]


for original in examples:
    a = list(original)
    wiggle_todo_ans(a)
    assert check_result(original, a)


# Bài 4
def is_wiggle_reverse(a):
    return all((a[i] > a[i + 1]) if i % 2 == 0 else (a[i] < a[i + 1]) for i in range(len(a) - 1))


def wiggle_sort_reverse(nums):
    t = sorted(nums, reverse=True)                               # gương: "lớn" thành "nhỏ"
    m = (len(nums) + 1) // 2
    nums[0::2] = t[:m][::-1]                                     # đỉnh (vị trí chẵn) lấy từ nửa lớn nhất
    nums[1::2] = t[m:][::-1]


rng = random.Random(25)
for _ in range(300):
    n = rng.randint(1, 30)
    nums = [4 - x for x in random_valid_input(n, 4, rng)]        # zigzag thường của x <=> zigzag đảo của 4 - x
    a = nums[:]
    wiggle_sort_reverse(a)
    assert sorted(a) == sorted(nums) and is_wiggle_reverse(a), (nums, a)


# Bài 5
def random_zigzag(n, hi, seed):
    rng = random.Random(seed)
    a = [rng.randint(0, hi - 1)]
    for i in range(1, n):
        a.append(rng.randint(a[-1] + 1, hi) if i % 2 == 1 else rng.randint(0, a[-1] - 1))
    rng.shuffle(a)
    return a


for seed in range(200):
    nums = random_zigzag(random.Random(seed).randint(1, 7), 3, seed)
    assert oracle_exists(nums), nums
print("Bài 5 OK")
print("PASS đáp án bài tập")

Bài 5 OK
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp, chia tại `⌈n/2⌉`, xen kẽ đảo ngược)

```python
class Solution:
    def wiggleSort(self, nums: List[int]) -> None:
        arr = sorted(nums)
        n = len(arr)
        i, j = (n - 1) >> 1, n - 1
        for k in range(n):
            if k % 2 == 0:
                nums[k] = arr[i]
                i -= 1
            else:
                nums[k] = arr[j]
                j -= 1
```

**Checklist**

- [ ] Nửa nhỏ có `⌈n/2⌉` phần tử, đặt vào **vị trí chẵn**; nửa lớn vào **vị trí lẻ**?
- [ ] **Đảo ngược** từng nửa (đọc từ cuối về đầu) để các bản của trung vị không kề nhau?
- [ ] Sửa `nums` **tại chỗ**, không trả giá trị; kết quả là **hoán vị** của đầu vào?
- [ ] Test bằng **tính hợp lệ** (zigzag nghiêm ngặt + cùng đa tập), không so cứng?
- [ ] Biết điều kiện tần suất chỉ là điều kiện cần (đề đảm bảo có đáp án) và hướng `O(n)` (quickselect + chỉ số ảo)?
- [ ] Đã thử `[4,5,5,6]`, `n` lẻ, hai giá trị mỗi giá trị `n/2` lần, `n = 5·10^4`?
- [ ] Skeleton đúng `class Solution` / `def wiggleSort(self, nums: List[int]) -> None`?